# Fine-tune the field detector (GPU notebook — PENDING USER RUN)

**This notebook has not been executed.** It was authored on a CPU-only machine with no GPU available (see `docs/edge-feasibility.md`). No numbers in this repo come from running this notebook — every cell that produces a metric is marked below, and none of those values exist yet.

**What this does**: fine-tunes a torchvision Faster R-CNN (MobileNetV3-Large-320-FPN, COCO-pretrained, BSD-3-Clause license — see `docs/data.md` "Detector license choice") on the synthetic USDOT/trailer_id/plate/seal dataset produced by `data_pipeline/synthetic/generate_synthetic.py`, and reports before/after mAP@0.5 on the SAME held-out synthetic test split.

**Why synthetic-only fine-tuning**: no public labeled dataset exists for USDOT/trailer_id/seal (see `docs/data.md`). Fine-tuning here can only improve accuracy *on synthetic data* — it says nothing about real gate imagery, and that gap is reported explicitly in `docs/feasibility.md`, not glossed over.

**How to run**: open in Google Colab, Runtime -> Change runtime type -> GPU (T4 is enough), then Run All. Clone/upload this repo first (cell 2).

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> GPU before running this notebook.'

In [ ]:
# Clone the repo (replace with your fork/branch if different) and cd into it.
!git clone https://github.com/HxrshRaj/dwarpal.git
%cd dwarpal
!pip install -q -r requirements.txt

In [ ]:
# Regenerate synthetic data on this machine (deterministic --seed, so this
# matches what the CPU-only benchmark run used) and build the YOLO-format index.
!python data_pipeline/synthetic/generate_synthetic.py --count 2000 --seed 42
!python data_pipeline/convert_to_yolo.py
!python data_pipeline/split_by_source.py

In [ ]:
import json
import sys
from pathlib import Path

import torch
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn, FasterRCNN_MobileNet_V3_Large_320_FPN_Weights

sys.path.insert(0, str(Path.cwd()))
from benchmarks.metrics import average_precision_at_iou, mean_average_precision

PROCESSED = Path('data_pipeline/processed')
CLASSES = ['truck', 'plate', 'usdot', 'trailer_id', 'seal']

class YoloFolderDataset(Dataset):
    def __init__(self, split):
        rows = [json.loads(l) for l in (PROCESSED / 'index_split.jsonl').read_text().splitlines()]
        self.rows = [r for r in rows if r['split'] == split and r['source'] == 'synthetic']

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img_path = PROCESSED / 'images' / row['image']
        lbl_path = PROCESSED / 'labels' / row['image'].replace('.png', '.txt')
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        boxes, labels = [], []
        for line in lbl_path.read_text().splitlines():
            cls_id, xc, yc, bw, bh = line.split()
            cls_id = int(cls_id)
            xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
            labels.append(cls_id + 1)  # 0 reserved for background
        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            'labels': torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
        }
        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        return img_t, target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = YoloFolderDataset('train')
test_ds = YoloFolderDataset('test')
print('train:', len(train_ds), 'test:', len(test_ds))

In [ ]:
# BEFORE: evaluate the off-the-shelf pretrained model (not fine-tuned) on the
# held-out synthetic test split, for each field class. This number IS
# reproducible on CPU too (see benchmarks/run_benchmarks.py) -- it is included
# here only so the before/after comparison lives in one notebook.

def evaluate(model, dataset, device):
    model.eval()
    per_class_ap = {c: [] for c in CLASSES}
    loader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = [i.to(device) for i in imgs]
            outputs = model(imgs)
            out, tgt = outputs[0], targets[0]
            for cls_idx, cls_name in enumerate(CLASSES, start=1):
                gt_boxes = tgt['boxes'][tgt['labels'] == cls_idx].tolist()
                gt_xywh = [[b[0], b[1], b[2]-b[0], b[3]-b[1]] for b in gt_boxes]
                pred_mask = out['labels'] == cls_idx
                pred_boxes = out['boxes'][pred_mask].tolist()
                pred_scores = out['scores'][pred_mask].tolist()
                pred_xywh = [([b[0], b[1], b[2]-b[0], b[3]-b[1]], s) for b, s in zip(pred_boxes, pred_scores)]
                if gt_xywh or pred_xywh:
                    per_class_ap[cls_name].append(average_precision_at_iou(pred_xywh, gt_xywh))
    return {c: mean_average_precision(v) for c, v in per_class_ap.items() if v}

device = 'cuda'
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT, num_classes=91)
# Replace the box predictor head for our 5 classes + background
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, len(CLASSES) + 1)
model.to(device)

before_metrics = evaluate(model, test_ds, device)
print('BEFORE fine-tuning (freshly-initialized head, COCO backbone):', before_metrics)

In [ ]:
# Fine-tune on the synthetic train split.
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=0.0005)
loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)

EPOCHS = 10  # adjust based on available GPU time
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for imgs, targets in loader:
        imgs = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f'epoch {epoch+1}/{EPOCHS} loss={epoch_loss/len(loader):.4f}')

In [ ]:
after_metrics = evaluate(model, test_ds, device)
print('AFTER fine-tuning:', after_metrics)
print('BEFORE:', before_metrics)

import json
Path('benchmarks/results').mkdir(parents=True, exist_ok=True)
Path('benchmarks/results/finetune_before_after.json').write_text(json.dumps({'before': before_metrics, 'after': after_metrics, 'epochs': EPOCHS, 'n_train': len(train_ds), 'n_test': len(test_ds)}, indent=2))
print('Wrote benchmarks/results/finetune_before_after.json -- copy this back into the repo and cite it in docs/benchmarks.md.')

In [ ]:
torch.save(model.state_dict(), 'training/finetuned_detector.pt')
dummy = torch.rand(1, 3, 320, 320).to(device)
model.eval()
torch.onnx.export(model, [dummy[0]], 'training/finetuned_detector.onnx', input_names=['images'], output_names=['boxes','labels','scores'], opset_version=17)
print('Saved weights + ONNX export. Download these from Colab and commit them (or attach to a release) before citing them in docs.')